# Using Credit Card Fraud Dataset

In [1]:
!pip install xgboost imbalanced-learn shap scikit-learn pandas numpy matplotli

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement matplotli (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for matplotli


In [2]:
!pip install xgboost imbalanced-learn shap scikit-learn pandas numpy matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Data & View Shape #

In [3]:
import pandas as pd

df = pd.read_csv('C:/Users/USER/OneDrive/Desktop/fraud-shield/notebooks/creditcard.csv')
print("Shape:", df.shape)
print(df.head())

Shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26 

## Check Fraud Positive Percentage

In [4]:
print(df['Class'].value_counts())
print("Fraud %:", round(df['Class'].mean() * 100, 3), "%")

Class
0    284315
1       492
Name: count, dtype: int64
Fraud %: 0.173 %


In [5]:
import numpy as np

# Extract hour from Time column
df['hour'] = (df['Time'] / 3600 % 24).astype(int)

# Log-transform Amount
df['amount_log'] = np.log1p(df['Amount'])

# Drop original Time and Amount
df = df.drop(columns=['Time', 'Amount'])

print("New shape:", df.shape)
print("New columns:", df.columns.tolist())

New shape: (284807, 31)
New columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'hour', 'amount_log']


## Separate Test set and Training Set

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Class'])   # everything except the answer
y = df['Class']                  # the answer (0 or 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Fraud in training set:", y_train.sum())
print("Fraud in test set:", y_test.sum())

Training rows: 227845
Testing rows: 56962
Fraud in training set: 394
Fraud in test set: 98


In [7]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

print("Before SMOTE — Fraud:", y_train.sum(), "| Legit:", (y_train==0).sum())
print("After SMOTE  — Fraud:", y_res.sum(), "| Legit:", (y_res==0).sum())

Before SMOTE — Fraud: 394 | Legit: 227451
After SMOTE  — Fraud: 227451 | Legit: 227451


## Train xgboost model

In [8]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_res, y_res)
print("Model trained!")

Model trained!


## Evaluate Model

In [9]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print("AUC-ROC Score:", round(roc_auc_score(y_test, y_prob), 4))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.37      0.87      0.52        98

    accuracy                           1.00     56962
   macro avg       0.69      0.93      0.76     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC Score: 0.9833


## Threshold Optimization

In [10]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Find threshold where both precision and recall are balanced
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print("Best threshold:", round(best_threshold, 4))

# Re-evaluate with new threshold
y_pred_new = (y_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_new, target_names=['Legit', 'Fraud']))

Best threshold: 0.9928
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.96      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.98      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962



## Save model

In [11]:
import os
import joblib

save_path = 'C:/Users/USER/OneDrive/Desktop/fraud-shield/backend/model'

joblib.dump(model, f'{save_path}/fraud_model.pkl')
joblib.dump(X.columns.tolist(), f'{save_path}/feature_names.pkl')
joblib.dump(float(best_threshold), f'{save_path}/threshold.pkl')

print("All files saved to backend/model/ ✅")
print("Files:", os.listdir(save_path))

All files saved to backend/model/ ✅
Files: ['feature_names.pkl', 'fraud_model.pkl', 'threshold.pkl']


## Train LightGBM 

In [13]:
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score

# Train LightGBM on same SMOTE-balanced data
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_res, y_res)
print("LightGBM trained!")

LightGBM trained!


## Compare both model

In [14]:
# XGBoost predictions
xgb_prob = model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob >= best_threshold).astype(int)

# LightGBM predictions
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
lgb_pred = (lgb_prob >= 0.5).astype(int)

print("=== XGBoost ===")
print(classification_report(y_test, xgb_pred, target_names=['Legit', 'Fraud']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_prob), 4))

print("=== LightGBM ===")
print(classification_report(y_test, lgb_pred, target_names=['Legit', 'Fraud']))
print("AUC-ROC:", round(roc_auc_score(y_test, lgb_prob), 4))

=== XGBoost ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.96      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.98      0.88      0.92     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC: 0.9833
=== LightGBM ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.36      0.89      0.51        98

    accuracy                           1.00     56962
   macro avg       0.68      0.94      0.76     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC: 0.9865


## Ensemble both model

In [15]:
import numpy as np

# Average both model scores
ensemble_prob = (xgb_prob + lgb_prob) / 2
ensemble_pred = (ensemble_prob >= best_threshold).astype(int)

print("=== Ensemble (XGBoost + LightGBM) ===")
print(classification_report(y_test, ensemble_pred, target_names=['Legit', 'Fraud']))
print("AUC-ROC:", round(roc_auc_score(y_test, ensemble_prob), 4))

=== Ensemble (XGBoost + LightGBM) ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.92      0.74      0.82        98

    accuracy                           1.00     56962
   macro avg       0.96      0.87      0.91     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC: 0.9854


## Save LightGBM Model

In [16]:
import joblib

save_path = 'C:/Users/USER/OneDrive/Desktop/fraud-shield/backend/model'
joblib.dump(lgb_model, f'{save_path}/lgb_model.pkl')
print("Saved lgb_model.pkl ✅")

Saved lgb_model.pkl ✅


In [17]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, lgb_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_lgb_threshold = thresholds[np.argmax(f1_scores)]

print("Best LightGBM threshold:", round(best_lgb_threshold, 4))

lgb_pred_new = (lgb_prob >= best_lgb_threshold).astype(int)
print(classification_report(y_test, lgb_pred_new, target_names=['Legit', 'Fraud']))

Best LightGBM threshold: 0.9874
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.88      0.81      0.84        98

    accuracy                           1.00     56962
   macro avg       0.94      0.90      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [18]:
save_path = 'C:/Users/USER/OneDrive/Desktop/fraud-shield/backend/model'
joblib.dump(lgb_model, f'{save_path}/lgb_model.pkl')
joblib.dump(float(best_lgb_threshold), f'{save_path}/lgb_threshold.pkl')

print("Saved lgb_model.pkl ✅")
print("Saved lgb_threshold.pkl ✅")
print("LightGBM threshold:", round(best_lgb_threshold, 4))

Saved lgb_model.pkl ✅
Saved lgb_threshold.pkl ✅
LightGBM threshold: 0.9874


# Another Dataset : Paysim

In [19]:
# Load PaySim dataset
paysim = pd.read_csv('PS_20174392719_1491204439457_log.csv')
print("Shape:", paysim.shape)
print("Columns:", paysim.columns.tolist())
print("\nTransaction types:")
print(paysim['type'].value_counts())
print("\nFraud distribution:")
print(paysim['isFraud'].value_counts())
print("Fraud %:", round(paysim['isFraud'].mean() * 100, 3), "%")

Shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64
Fraud %: 0.129 %


## Feature Engineering

In [21]:
# Keep only TRANSFER and CASH_OUT — fraud only happens here
paysim = paysim[paysim['type'].isin(['TRANSFER', 'CASH_OUT'])]
print("After filtering:", paysim.shape)

# Feature engineering
paysim['hour'] = paysim['step'] % 24
paysim['amount_log'] = np.log1p(paysim['amount'])

# Balance difference features
paysim['orig_balance_diff'] = paysim['oldbalanceOrg'] - paysim['newbalanceOrig']
paysim['dest_balance_diff'] = paysim['newbalanceDest'] - paysim['oldbalanceDest']

# Suspicious pattern — money moved but destination balance unchanged
paysim['balance_mismatch'] = (
    (paysim['orig_balance_diff'] - paysim['amount']).abs() > 1
).astype(int)

# Transaction type encoded
paysim['is_transfer'] = (paysim['type'] == 'TRANSFER').astype(int)

# Final features
paysim_features = paysim[[
    'hour', 'amount_log', 'orig_balance_diff',
    'dest_balance_diff', 'balance_mismatch', 'is_transfer'
]]
paysim_target = paysim['isFraud']

print("Features shape:", paysim_features.shape)
print("Fraud cases:", paysim_target.sum())
print("Fraud %:", round(paysim_target.mean() * 100, 3), "%")

After filtering: (2770409, 11)
Features shape: (2770409, 6)
Fraud cases: 8213
Fraud %: 0.296 %


## Sovle Imbalence dataset with SMOTE

In [22]:
# Split data
X_pay = paysim_features
y_pay = paysim_target

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pay, y_pay, test_size=0.2, random_state=42, stratify=y_pay
)

print("Train size:", X_train_p.shape[0])
print("Test size:", X_test_p.shape[0])
print("Fraud in train:", y_train_p.sum())

# SMOTE — only on training data
sm2 = SMOTE(random_state=42)
X_res_p, y_res_p = sm2.fit_resample(X_train_p, y_train_p)

print("\nAfter SMOTE:")
print("Fraud:", y_res_p.sum(), "| Legit:", (y_res_p==0).sum())

Train size: 2216327
Test size: 554082
Fraud in train: 6570

After SMOTE:
Fraud: 2209757 | Legit: 2209757


## Train XGBoost model using Paysim Dataset

In [25]:
# Train XGBoost on PaySim data
paysim_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42
)

paysim_model.fit(X_res_p, y_res_p)
print("PaySim model trained!")

PaySim model trained!


## Evaluate 

In [26]:
# Evaluate PaySim model
y_pred_p = paysim_model.predict(X_test_p)
y_prob_p = paysim_model.predict_proba(X_test_p)[:, 1]

print(classification_report(y_test_p, y_pred_p, target_names=['Legit', 'Fraud']))
print("AUC-ROC:", round(roc_auc_score(y_test_p, y_prob_p), 4))

              precision    recall  f1-score   support

       Legit       1.00      0.95      0.97    552439
       Fraud       0.05      0.96      0.10      1643

    accuracy                           0.95    554082
   macro avg       0.53      0.96      0.54    554082
weighted avg       1.00      0.95      0.97    554082

AUC-ROC: 0.9929


## Optimize Threshold

In [27]:
# Find best threshold for PaySim model
precisions_p, recalls_p, thresholds_p = precision_recall_curve(y_test_p, y_prob_p)
f1_scores_p = 2 * (precisions_p * recalls_p) / (precisions_p + recalls_p + 1e-8)
best_paysim_threshold = thresholds_p[np.argmax(f1_scores_p)]

print("Best PaySim threshold:", round(best_paysim_threshold, 4))

# Re-evaluate with new threshold
y_pred_p_new = (y_prob_p >= best_paysim_threshold).astype(int)
print(classification_report(y_test_p, y_pred_p_new, target_names=['Legit', 'Fraud']))
print("AUC-ROC:", round(roc_auc_score(y_test_p, y_prob_p), 4))

Best PaySim threshold: 0.9908
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    552439
       Fraud       0.96      0.78      0.86      1643

    accuracy                           1.00    554082
   macro avg       0.98      0.89      0.93    554082
weighted avg       1.00      1.00      1.00    554082

AUC-ROC: 0.9929


## Save Paysim Model

In [28]:
save_path = 'C:/Users/USER/OneDrive/Desktop/fraud-shield/backend/model'
joblib.dump(paysim_model, f'{save_path}/paysim_model.pkl')
joblib.dump(paysim_features.columns.tolist(), f'{save_path}/paysim_feature_names.pkl')
joblib.dump(float(best_paysim_threshold), f'{save_path}/paysim_threshold.pkl')

print("Saved paysim_model.pkl ✅")
print("Saved paysim_feature_names.pkl ✅")
print("Saved paysim_threshold.pkl ✅")

Saved paysim_model.pkl ✅
Saved paysim_feature_names.pkl ✅
Saved paysim_threshold.pkl ✅
